## 1. 기본 패키지 불러오기
EDA에서 필요한 최소 패키지만 불러옵니다. 이 단계는 데이터를 바꾸지 않고, 화면에 확인 결과를 보여주기 위한 준비 단계입니다.

In [ ]:
# 기본 패키지
from pathlib import Path
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. 팀 공통 Google Drive 경로 설정
Colab에서 바로 실행할 수 있도록 기존 Drive 경로를 그대로 사용합니다. 여기서는 경로가 맞는지 확인만 합니다.

In [ ]:
from pathlib import Path

# ============================================================
# Google Drive 경로 설정
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
#
# Drive 폴더 구조:
# Graph_AML/
# └── data/
#     ├── code/
#     ├── raw/
#     ├── processed/
#     │   ├── clean_base/
#     │   └── ml_features/
#     └── graph/
#         └── gnn_inputs/
#
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# 모든 산출물은 Google Drive에 저장합니다.
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

CODE_DIR = DRIVE_DATA_DIR / "code"
RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"
GRAPH_DIR = DRIVE_DATA_DIR / "graph"

CLEAN_BASE_DIR = PROCESSED_DIR / "clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = GRAPH_DIR / "gnn_inputs"


# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
CLEAN_BASE_DIR = CLEAN_BASE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    CODE_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    GRAPH_DIR,
    CLEAN_BASE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("CLEAN_BASE_DIR :", CLEAN_BASE_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)


# 01번은 원본 거래 파일만 읽습니다.
def resolve_raw_transaction_path(raw_dir):
    raw_dir = Path(raw_dir)
    candidate_names = [
        "HI-Small_Trans.csv", "HI-Small_Trans.parquet",
        "LI-Small_Trans.csv", "LI-Small_Trans.parquet",
        "HI-Medium_Trans.csv", "LI-Medium_Trans.csv",
        "HI-Large_Trans.csv", "LI-Large_Trans.csv",
        "transactions.csv", "Transactions.csv", "trans.csv",
    ]
    for name in candidate_names:
        p = raw_dir / name
        if p.exists():
            return p

    trans_like = [
        p for p in raw_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in [".csv", ".parquet", ".pq"]
        and ("trans" in p.name.lower() or "transaction" in p.name.lower())
    ]
    if trans_like:
        return sorted(trans_like)[0]

    raise FileNotFoundError(
        "거래 파일을 찾지 못했습니다. Graph_AML/data/raw 폴더에 거래 CSV를 넣어주세요. "
        f"RAW_DIR={raw_dir}"
    )

RAW_DATA_PATH = resolve_raw_transaction_path(RAW_DIR)

print("RAW_DATA_PATH:", RAW_DATA_PATH, "exists=", RAW_DATA_PATH.exists())


## 3. 원본 데이터 로드
원본 거래 파일을 읽어옵니다. 01번 EDA에서는 파일을 저장하지 않고, 원본 구조와 값의 상태를 화면 출력으로만 확인합니다.

In [ ]:
# ============================================================
# 원본 데이터 로드
# ============================================================
# EDA 목적이므로 원본 컬럼을 그대로 읽습니다.
# 대용량 파일이면 SAMPLE_ROWS 값을 조정해서 먼저 확인할 수 있습니다.

SAMPLE_ROWS = None  # 예: 100_000. 전체를 보려면 None.

if RAW_DATA_PATH.suffix.lower() == ".csv":
    df_raw = pd.read_csv(RAW_DATA_PATH, nrows=SAMPLE_ROWS)
elif RAW_DATA_PATH.suffix.lower() in [".parquet", ".pq"]:
    df_raw = pd.read_parquet(RAW_DATA_PATH)
    if SAMPLE_ROWS is not None:
        df_raw = df_raw.head(SAMPLE_ROWS)
else:
    raise ValueError(f"지원하지 않는 파일 형식입니다: {RAW_DATA_PATH.suffix}")

print("shape:", df_raw.shape)
display(df_raw.head(3))

## 4. 컬럼 구조 확인
원본 컬럼명과 데이터 타입을 확인합니다. 여기서 보이는 `Account`/`Account.1` 같은 쌍은 중복이 아니라 송신자/수신자 구조를 의미합니다.

In [ ]:
# ============================================================
# 컬럼 구조 확인
# ============================================================
# 원본 컬럼명, dtype, 결측치 수를 한 번에 확인합니다.

schema_review = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "null_count": [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    "null_ratio": [float(df_raw[c].isna().mean()) for c in df_raw.columns],
    "n_unique": [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns],
})

display(schema_review)

print("컬럼 수:", len(df_raw.columns))
print("컬럼 목록:")
for c in df_raw.columns:
    print("-", c)

## 컬럼이 2개씩 있는 것처럼 보이는 이유

원본 IBM AML 데이터에는 송신자와 수신자를 한 행에 함께 표현하기 때문에 비슷한 의미의 컬럼이 쌍으로 존재합니다.

예시
- `From Bank` / `To Bank`: 송신 은행 / 수신 은행
- `Account` / `Account.1`: 송신 계좌 / 수신 계좌
- `Amount Paid` / `Amount Received`: 송금액 / 수취액
- `Payment Currency` / `Receiving Currency`: 송금 통화 / 수취 통화

따라서 이 컬럼들은 단순 중복이라기보다 **거래 방향을 표현하기 위한 sender/receiver 쌍**입니다.  
preprocessing 단계에서는 팀원이 ML/GNN에서 일관되게 쓰도록 `sender_*`, `receiver_*` 같은 표준 컬럼명을 추가합니다. 원본 컬럼을 바로 삭제하지 않는 이유는 추적과 검증을 쉽게 하기 위해서입니다.

## 5. 결측치와 기본 통계 확인
결측치가 있는 컬럼과 기본 통계를 확인합니다. 이 결과는 전처리에서 어떤 검증이 필요한지 판단하는 근거가 됩니다.

In [ ]:
# ============================================================
# 유사 컬럼 쌍 확인
# ============================================================
# 컬럼이 실제 중복인지, 송신/수신 역할이 다른 컬럼인지 확인합니다.

pairs = [
    ("From Bank", "To Bank"),
    ("Account", "Account.1"),
    ("Amount Paid", "Amount Received"),
    ("Payment Currency", "Receiving Currency"),
]

for a, b in pairs:
    if a in df_raw.columns and b in df_raw.columns:
        same_ratio = (df_raw[a].astype(str) == df_raw[b].astype(str)).mean()
        print(f"{a} vs {b}")
        print(f"- same_ratio: {same_ratio:.4f}")
        print(f"- {a} unique: {df_raw[a].nunique(dropna=True):,}")
        print(f"- {b} unique: {df_raw[b].nunique(dropna=True):,}")
        print()
    else:
        print(f"skip: {a}, {b} 중 하나가 없습니다.")

## 6. 라벨 분포 확인
`Is_Laundering`의 0/1 분포를 확인합니다. 라벨이 불균형할 가능성이 크기 때문에 이후 모델 평가에서는 단순 accuracy보다 F1, Precision, Recall, PR-AUC 등을 함께 봐야 합니다.

In [ ]:
# ============================================================
# 라벨 분포 확인
# ============================================================
# laundering 라벨이 얼마나 희소한지 확인합니다.

label_candidates = ["Is Laundering", "Is_Laundering", "is_laundering", "label", "Label", "target"]
label_col = next((c for c in label_candidates if c in df_raw.columns), None)

if label_col is None:
    print("라벨 컬럼을 찾지 못했습니다.")
else:
    label_dist = (
        df_raw[label_col]
        .value_counts(dropna=False)
        .rename_axis("label")
        .reset_index(name="row_count")
    )
    label_dist["ratio"] = label_dist["row_count"] / len(df_raw)
    display(label_dist)
    print("해석: laundering=1 비율이 낮으면 일반적인 AML 데이터처럼 클래스 불균형이 큰 상태입니다.")

## 7. 시간대 분포 확인
거래가 특정 시간대에 몰리는지 확인합니다. 시간 피처를 넣을지 말지 판단하기 위한 탐색 단계이며, 이 노트북에서는 결론을 확정하지 않습니다.

In [ ]:
# ============================================================
# timestamp 파싱 및 시간대 분포 확인
# ============================================================
# 시간 피처가 유의미한지 보기 위한 기초 EDA입니다.

time_candidates = ["Timestamp", "timestamp", "Time", "time", "DateTime", "datetime"]
time_col = next((c for c in time_candidates if c in df_raw.columns), None)

if time_col is None:
    print("timestamp 컬럼을 찾지 못했습니다.")
else:
    ts = pd.to_datetime(df_raw[time_col], errors="coerce")
    print("timestamp parse null:", int(ts.isna().sum()))
    print("timestamp range:", ts.min(), "~", ts.max())

    tmp = pd.DataFrame({"hour": ts.dt.hour})
    if label_col is not None:
        tmp["label"] = pd.to_numeric(df_raw[label_col], errors="coerce")
    hourly = tmp.groupby("hour", dropna=False).agg(
        tx_count=("hour", "size"),
        laundering_ratio=("label", "mean") if label_col is not None else ("hour", "size"),
    ).reset_index()
    display(hourly)

    ax = hourly.dropna(subset=["hour"]).plot(x="hour", y="tx_count", kind="bar", figsize=(10, 4), legend=False)
    ax.set_title("Transaction Count by Hour")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Transaction Count")
    plt.tight_layout()
    plt.show()

    if label_col is not None:
        ax = hourly.dropna(subset=["hour"]).plot(x="hour", y="laundering_ratio", kind="line", marker="o", figsize=(10, 4), legend=False)
        ax.set_title("Laundering Ratio by Hour")
        ax.set_xlabel("Hour")
        ax.set_ylabel("Laundering Ratio")
        plt.tight_layout()
        plt.show()

## 8. 금액 분포 확인
금액은 평균만 보면 놓치는 부분이 있을 수 있어 분위수, 분산, 이상치 관점으로 함께 확인합니다. 라벨별 차이가 보이면 이후 ML 피처 후보로 검토합니다.

In [ ]:
# ============================================================
# 금액 분포 확인
# ============================================================
# 분포가 비슷해 보이는지/분산 차이가 있는지 숫자로 같이 확인합니다.

amount_candidates = ["Amount Paid", "Amount_Paid", "amount_paid", "Amount Received", "Amount_Received", "amount"]
amount_col = next((c for c in amount_candidates if c in df_raw.columns), None)

if amount_col is None:
    print("amount 컬럼을 찾지 못했습니다.")
else:
    amount = pd.to_numeric(df_raw[amount_col], errors="coerce")
    desc = amount.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame(name=amount_col)
    display(desc)

    if label_col is not None:
        tmp = pd.DataFrame({
            "amount": amount,
            "label": pd.to_numeric(df_raw[label_col], errors="coerce"),
        })
        by_label = tmp.groupby("label")["amount"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
        display(by_label)
        print("해석: 평균만 보지 말고 median, 95%, 99%, std를 같이 봐야 라벨별 분산 차이를 설명할 수 있습니다.")

    ax = np.log1p(amount.dropna()).plot(kind="hist", bins=50, figsize=(10, 4))
    ax.set_title(f"log1p({amount_col}) Distribution")
    ax.set_xlabel("log1p(amount)")
    plt.tight_layout()
    plt.show()